# Exercise 2.7: Transforming and Merging (Angola IEA and INE trade)

Two sources, two jobs. The survey cleaned in 2.6 becomes analysis ready through
custom functions and `apply`. The INE trade workbooks are then loaded, merged and
stacked.

**PT:** Duas fontes, dois trabalhos. O inquerito limpo em 2.6 torna-se pronto
para analise com funcoes proprias e `apply`. Depois carregamos, juntamos e
empilhamos os ficheiros de comercio do INE.

> **Pipeline:** run 2.6 first. Reads `10_cleaned/` and `0_raw/angola`, writes
> `20_processed/`.

### Path Setup (run first)

**PT:** Configuracao dos caminhos.

In [ ]:
import os

import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw/angola'
DATA_CLEAN_DIR = '../../data/10_cleaned'
DATA_PROC_DIR = '../../data/20_processed'

TRADE_DIR = 'international_trade'

clean_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_clean.csv')
trade_dir = os.path.join(DATA_RAW_DIR, TRADE_DIR)

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Trade workbooks available / Ficheiros de comercio disponiveis:')
for name in sorted(os.listdir(trade_dir)):
    print('  ', name)

---

# Part A: the survey

## Task 1: Load the survey and restore its types

CSV forgets dtypes. Notebook 2.6 saved them in the codebook, so read that file
first and use it.

**What to do:** read the codebook, build two dictionaries from it, one mapping
`new_name` to `description` and one mapping `new_name` to `dtype_final`, then
pass the dtypes to `read_csv`. Datetime columns cannot go through `dtype=`, so
they are separated into `parse_dates`.

**PT:** O CSV esquece os tipos. O caderno 2.6 gravou-os no dicionario.

**O que fazer:** leia o dicionario, construa dois dicionarios a partir dele, um
de `new_name` para `description` e outro de `new_name` para `dtype_final`, e
passe os tipos ao `read_csv`. As colunas de data nao podem ir em `dtype=`, por
isso vao em `parse_dates`.

In [ ]:
codebook_path = os.path.join(DATA_CLEAN_DIR, 'angola_iea_2025q4_codebook.csv')
codebook_df = pd.read_csv(codebook_path)

descriptions =   # your code here: new_name -> description
dtypes =   # your code here: new_name -> dtype_final

# dtype= cannot build a datetime, those columns need parse_dates instead
# dtype= nao constroi datas, essas colunas precisam de parse_dates
date_cols = [col for col, kind in dtypes.items() if kind.startswith('datetime')]
read_dtypes = {col: kind for col, kind in dtypes.items()
               if not kind.startswith('datetime')}

df = pd.read_csv(clean_path, dtype=read_dtypes, parse_dates=date_cols)

print('Survey:', df.shape)
print(df[['household_id', 'age', 'job_start_year', 'interview_date']].dtypes)

**Questions:**

- How many rows and columns did you load, and did every dtype come back?
- Which column could not be restored through `dtype=`, and why not?

**PT:** Quantas linhas e colunas carregou, e todos os tipos voltaram? Que coluna
nao pode ser restaurada por `dtype=`, e porque?

---

## Task 2: Write a function and apply it to one column

`apply` on a Series runs your function once per value. Use it when the rule needs
branching that a single expression cannot express clearly.

**What to do:** complete `age_band` so it returns `'Child'` under 15, `'Youth'`
under 25, `'Adult'` under 65 and `'Elderly'` otherwise, then apply it to `age` and
store the result in a new column `age_band`.

**PT:** `apply` numa Serie corre a funcao para cada valor.

**O que fazer:** complete `age_band` para devolver `'Child'` abaixo de 15,
`'Youth'` abaixo de 25, `'Adult'` abaixo de 65 e `'Elderly'` nos restantes casos.
Depois aplique a coluna `age` e guarde em `age_band`.

In [ ]:
def age_band(age):
    """ILO oriented age band for one person / Faixa etaria para uma pessoa."""
    # your code here: band at 15, 25 and 65
    # o seu codigo aqui: faixas em 15, 25 e 65
    return


df['age_band'] = df['age'].apply(age_band)
df['age_band'].value_counts()

**Questions:**

- How many people fall in each band?
- Where do the thresholds 15 and 65 come from?

**PT:** Quantas pessoas em cada faixa? De onde vem os limiares 15 e 65?

---

## Task 3: Write a function that reads several columns at once

`apply(axis=1)` passes a whole row to your function, so it can read many columns
together. Labour force status is the natural case: it depends on eight answers,
and no single column expression can express it.

Because 2.6 kept the value labels, the answers are the Portuguese words `Sim` and
`Nao`, so the function reads almost like the questionnaire.

**What to do:** complete `labour_force_status` following the ILO rule:

1. under 15 years old, return `'Outside labour force'`
2. answered `Sim` to any of `worked_for_pay`, `worked_own_account` or
   `absent_from_job`, return `'Employed'`
3. answered `Sim` to `sought_job` **or** `sought_business`, **and** `Sim` to
   `available_last_week` **or** `available_next_2weeks`, return `'Unemployed'`
4. otherwise return `'Outside labour force'`

Then apply it with `axis=1` and store the result in `lf_status`.

**PT:** `apply(axis=1)` passa a linha inteira, por isso a funcao pode ler varias
colunas. Como 2.6 manteve as etiquetas, as respostas sao `Sim` e `Nao`.

**O que fazer:** complete `labour_force_status` segundo a regra da OIT: menos de
15 anos, fora da forca de trabalho; `Sim` a qualquer uma das tres perguntas de
trabalho, empregado; `Sim` a procura **e** `Sim` a disponibilidade, desempregado;
caso contrario, fora da forca de trabalho. Aplique com `axis=1` e guarde em
`lf_status`.

In [ ]:
def labour_force_status(row):
    """ILO status for one person / Situacao perante o trabalho de uma pessoa."""
    if row['age'] < 15:
        return 'Outside labour force'

    worked = (row['worked_for_pay'], row['worked_own_account'], row['absent_from_job'])
    if 'Sim' in worked:
        return 'Employed'

    searched =   # your code here: Sim to sought_job or sought_business
    available =   # your code here: Sim to either availability question
    if searched and available:
        return 'Unemployed'

    return 'Outside labour force'


df['lf_status'] = df.apply(labour_force_status, axis=1)
df['lf_status'].value_counts()

**Questions:**

- How many are employed, unemployed and outside the labour force?
- Try the availability test with `available_next_2weeks` alone. What rate do you
  get, and why is it wrong?

**PT:** Quantos empregados, desempregados e fora da forca de trabalho? Experimente
testar a disponibilidade so com `available_next_2weeks`: que taxa obtem, e porque
esta errada?

---

## Task 4: Weight the result

Each person represents many Angolans, and the `weight` column says how many. An
unweighted rate describes the sample; a weighted rate describes the country.

**What to do:** complete `weighted_share` so it returns the weighted percentage
of the population picked out by a boolean mask, then use it for the unemployment
rate, which is over the labour force, and the participation rate, which is over
the working age population.

**PT:** Cada pessoa representa muitos angolanos, e a coluna `weight` diz quantos.

**O que fazer:** complete `weighted_share` para devolver a percentagem ponderada
selecionada por uma mascara, e use-a para a taxa de desemprego, sobre a forca de
trabalho, e a taxa de atividade, sobre a populacao em idade ativa.

In [ ]:
def weighted_share(mask, weights):
    """Weighted percentage selected by `mask` / Percentagem ponderada."""
    # your code here: weighted share of the mask, as a percentage
    # o seu codigo aqui: percentagem ponderada da mascara
    return


weight = df['weight']
in_labour_force = df['lf_status'].isin(['Employed', 'Unemployed'])
working_age = df['age'] >= 15

unemployment = weighted_share(df['lf_status'] == 'Unemployed', weight[in_labour_force])
participation = weighted_share(in_labour_force, weight[working_age])

print(f'Unemployment rate:  {unemployment:5.1f}%')
print(f'Participation rate: {participation:5.1f}%')

**Questions:**

- What are the unemployment and participation rates?
- Is this the strict or the relaxed definition?

**PT:** Quais sao as taxas de desemprego e de atividade? Esta e a definicao
estrita ou a alargada?

---

# Part B: the trade workbooks

## Task 5: Load a trade sheet

INE publishes the trade data as spreadsheets made for human readers: two title
rows above the header, a blank row, a `Total Geral` row, the data, and a source
footer at the bottom.

Three steps clean that up, and you will repeat them for a second workbook later:

1. `skiprows=2` so the real header becomes the header
2. strip the line break out of names like `Ano\n2004`
3. keep only rows where `País` is filled in, which drops the blank row, the total
   and the footer in one go

**What to do:** load the export sheet and the import sheet of the partner
countries workbook, applying those three steps to each.

**PT:** O INE publica os dados em folhas feitas para leitura humana: duas linhas
de titulo, o cabecalho, uma linha vazia, o `Total Geral`, os dados, e o rodape.

**O que fazer:** carregue a folha de exportacoes e a de importacoes aplicando os
tres passos: `skiprows=2`, limpar a quebra de linha nos nomes, e manter so as
linhas com `País` preenchido.

In [ ]:
PARTNERS_FILE = 'Comercio Externo de Bens por Países Parceiros.xlsx'
partners_path = os.path.join(trade_dir, PARTNERS_FILE)

exports = pd.read_excel(partners_path, sheet_name='Exportação por Países (USD)',
                        skiprows=2)
exports.columns = exports.columns.str.replace('\n', ' ', regex=False).str.strip()
exports = exports[exports['País'].notna()]

print('exports:', exports.shape)
exports[['Código', 'País', 'Ano 2025']].head()

In [ ]:
imports = pd.read_excel(partners_path, sheet_name='Importação por Países (USD)',
                        skiprows=2)
imports.columns = imports.columns.str.replace('\n', ' ', regex=False).str.strip()
imports = imports[imports['País'].notna()]

print('imports:', imports.shape)
imports[['Código', 'País', 'Ano 2025']].head()

**Questions:**

- How many rows does each sheet give? What is the extra one that is not a country?
- Why filter on `País` rather than dropping rows by position?
- What unit are the values in, and where does the file say so?

**PT:** Quantas linhas tem cada folha? Qual e a que nao e um pais? Porque filtrar
por `País` em vez de remover linhas por posicao? Em que unidade estao os valores?

---

## Task 6: Merge the two flows and classify each partner

Both tables have one row per country, so this is a one to one merge and
`validate` should say so.

**What to do:** merge exports and imports on the country code, keeping both
sides, with `indicator=True` and `validate='one_to_one'`. Then compute
`balance_thousand_usd` as exports minus imports, and complete `partner_profile`,
which needs both columns at once and therefore runs with `axis=1`.

**PT:** As duas tabelas tem uma linha por pais, por isso a juncao e um para um.

**O que fazer:** junte exportacoes e importacoes pelo codigo do pais, com
`indicator=True` e `validate='one_to_one'`. Depois calcule
`balance_thousand_usd` e complete `partner_profile`, que precisa das duas colunas
ao mesmo tempo e por isso corre com `axis=1`.

In [ ]:
YEAR = 'Ano 2025'

trade = pd.merge(
    exports[['Código', 'País', YEAR]].rename(columns={YEAR: 'exports_thousand_usd'}),
    imports[['Código', YEAR]].rename(columns={YEAR: 'imports_thousand_usd'}),
    # your code here: on the code column, outer, indicator, validate one to one
).rename(columns={'Código': 'country_code', 'País': 'country_name'})

print(trade['_merge'].value_counts())
trade = trade.drop(columns='_merge')
print('Merged:', trade.shape)

In [ ]:
def partner_profile(row):
    """Angola's 2025 relationship with one partner / Relacao com um parceiro."""
    sold = row['exports_thousand_usd']
    bought = row['imports_thousand_usd']

    if pd.isna(sold) or pd.isna(bought):
        return 'Incomplete'
    if sold + bought < 1000:
        return 'Negligible'
    # your code here: 'Angola mainly sells' when exports more than double
    # imports, 'Angola mainly buys' when the reverse, otherwise 'Two way'
    return


trade['balance_thousand_usd'] = (trade['exports_thousand_usd'].fillna(0)
                                 - trade['imports_thousand_usd'].fillna(0))
trade['profile'] = trade.apply(partner_profile, axis=1)

print(trade['profile'].value_counts())

In [ ]:
print('Largest surpluses / Maiores excedentes:')
print(trade.nlargest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))
print()
print('Largest deficits / Maiores defices:')
print(trade.nsmallest(5, 'balance_thousand_usd')[
    ['country_name', 'exports_thousand_usd', 'imports_thousand_usd', 'profile']
].to_string(index=False))

**Questions:**

- Did every partner match, and what does `validate='one_to_one'` promise?
- Which partners show the largest surplus and deficit?
- What would happen to the profiles without the `Negligible` threshold?

**PT:** Todos os parceiros corresponderam? Que parceiros tem maior excedente e
defice? O que aconteceria sem o limiar `Negligible`?

---

## Task 7: Handle a hierarchy before summing it

The economic categories workbook codes a tree in the length of the code: `1` is a
section, `11` a group inside it, `111` a subgroup inside that. Summing the column
adds every level together and gives the wrong answer.

**What to do:** load this workbook the same way as before, but filter on
`Descrição` instead of `País`. Keep the published `Total Geral` value aside as a
check. Then complete `cgce_level`, which returns the length of the code, apply it
to build a `level` column, and compare the sum of every row against the sum of
level 1 only.

**PT:** O ficheiro de categorias economicas codifica uma arvore no comprimento do
codigo: `1` seccao, `11` grupo, `111` subgrupo. Somar a coluna soma todos os
niveis.

**O que fazer:** carregue este ficheiro como antes, mas filtrando por
`Descrição`. Guarde o `Total Geral` publicado como verificacao. Depois complete
`cgce_level`, aplique-a para criar a coluna `level`, e compare a soma de todas as
linhas com a soma apenas do nivel 1.

In [ ]:
CGCE_FILE = 'Comercio Externo de Bens por Grandes Categorias Económicas.xlsx'
cgce_path = os.path.join(trade_dir, CGCE_FILE)

cgce = pd.read_excel(cgce_path, sheet_name='Export Cat. Económica (USD)', skiprows=2)
cgce.columns = cgce.columns.str.replace('\n', ' ', regex=False).str.strip()
cgce = cgce[cgce['Descrição'].notna()]
cgce['CGCE'] = cgce['CGCE'].astype('string')

# The published total, kept aside as the check / O total publicado, para verificar
published_total = cgce.loc[cgce['Descrição'] == 'Total Geral', YEAR].iloc[0]
cgce = cgce[cgce['CGCE'].notna()]

print('Published Total Geral:', f'{published_total:,.0f}')
cgce[['CGCE', 'Descrição', YEAR]].head()

In [ ]:
def cgce_level(code):
    """Depth in the CGCE tree: 1 section, 2 group, 3 subgroup.

    Profundidade na arvore CGCE: 1 seccao, 2 grupo, 3 subgrupo.
    """
    # your code here: the length of the code / o comprimento do codigo
    return


cgce['level'] = cgce['CGCE'].apply(cgce_level)
print(cgce['level'].value_counts().sort_index())

In [ ]:
naive =   # your code here: sum every row
sections_only =   # your code here: sum only the level 1 rows

print(f'Published total:     {published_total:15,.0f}')
print(f'Sum of every row:    {naive:15,.0f}  <- {naive / published_total:.2f}x')
print(f'Sum of level 1 only: {sections_only:15,.0f}')
print()
print('Level 1 matches the published total:',
      bool(abs(sections_only - published_total) < 1))

**Questions:**

- How many rows sit at each level of the tree?
- Compare the sum of every row with the published `Total Geral`. What is the
  ratio, and why is it exactly that?
- Which rows do you sum to reproduce the published figure?

**PT:** Quantas linhas em cada nivel? Compare a soma de todas as linhas com o
`Total Geral` publicado: qual e a razao e porque e exatamente essa? Que linhas
deve somar?

---

## Task 8: Stack the two flows

Merging combines columns, appending combines rows. Exports and imports have the
same shape, so a long format with a `flow` column is easier to group and chart.

**What to do:** take the code, country and 2025 value from each table, add a
`flow` column saying which is which **before** stacking, then concatenate them
with `ignore_index=True` and rename the columns.

**PT:** Juntar combina colunas, empilhar combina linhas.

**O que fazer:** tire o codigo, o pais e o valor de 2025 de cada tabela, adicione
uma coluna `flow` **antes** de empilhar, e concatene com `ignore_index=True`.

In [ ]:
long_exports =   # your code here: the three columns, plus flow='Export'
long_imports =   # your code here: the three columns, plus flow='Import'

flows = pd.concat([long_exports, long_imports], ignore_index=True)
flows = flows.rename(columns={'Código': 'country_code', 'País': 'country_name',
                              YEAR: 'value_thousand_usd'})

print('Stacked:', flows.shape)
print(flows.groupby('flow')['value_thousand_usd'].sum().round(0))

**Questions:**

- How many rows does the stacked table have?
- Why add the `flow` column before the concat rather than after?

**PT:** Quantas linhas tem a tabela empilhada? Porque adicionar a coluna `flow`
antes do concat?

---

## Task 9: Save

**What to do:** write the three tables to `20_processed/` with `index=False`.

**PT:** **O que fazer:** grave as tres tabelas em `20_processed/` com
`index=False`.

In [ ]:
os.makedirs(DATA_PROC_DIR, exist_ok=True)

survey_out = os.path.join(DATA_PROC_DIR, 'angola_iea_2025q4_features.csv')
trade_out = os.path.join(DATA_PROC_DIR, 'angola_trade_partners.csv')
flows_out = os.path.join(DATA_PROC_DIR, 'angola_trade_flows.csv')

# your code here: write the three frames with index=False
# o seu codigo aqui: gravar as tres tabelas com index=False

print('survey:', df.shape, '| trade:', trade.shape, '| flows:', flows.shape)

**Questions:**

- Which columns did the survey gain, and which function produced each?

**PT:** Que colunas ganhou o inquerito, e que funcao produziu cada uma?